# Lesson 01 — Centroid Tracking from Scratch

## Why This Lesson
Building a tracker from scratch forces you to understand what tracking actually is:
associating detections in frame N with detections in frame N-1 using spatial proximity.

```python
import cv2
import numpy as np
from scipy.spatial import distance as dist
import matplotlib.pyplot as plt

class CentroidTracker:
    def __init__(self, max_disappeared=30, max_distance=80):
        self.next_id      = 0
        self.objects      = {}   # id -> centroid
        self.disappeared  = {}   # id -> frames missing
        self.max_gone     = max_disappeared
        self.max_dist     = max_distance

    def register(self, c):
        self.objects[self.next_id]     = c
        self.disappeared[self.next_id] = 0
        self.next_id += 1

    def deregister(self, oid):
        del self.objects[oid]
        del self.disappeared[oid]

    def update(self, rects):
        if not rects:
            for oid in list(self.disappeared):
                self.disappeared[oid] += 1
                if self.disappeared[oid] > self.max_gone:
                    self.deregister(oid)
            return self.objects

        new_centroids = np.array([(x+w//2, y+h//2) for x,y,w,h in rects])

        if not self.objects:
            for c in new_centroids: self.register(c)
            return self.objects

        obj_ids  = list(self.objects.keys())
        obj_cents= np.array(list(self.objects.values()))

        D = dist.cdist(obj_cents, new_centroids)
        rows = D.min(axis=1).argsort()
        cols = D.argmin(axis=1)[rows]

        used_r, used_c = set(), set()
        for r,c in zip(rows, cols):
            if r in used_r or c in used_c: continue
            if D[r,c] > self.max_dist: continue
            oid = obj_ids[r]
            self.objects[oid]     = new_centroids[c]
            self.disappeared[oid] = 0
            used_r.add(r); used_c.add(c)

        for r in set(range(len(obj_ids))) - used_r:
            self.disappeared[obj_ids[r]] += 1
            if self.disappeared[obj_ids[r]] > self.max_gone:
                self.deregister(obj_ids[r])

        for c in set(range(len(new_centroids))) - used_c:
            self.register(new_centroids[c])

        return self.objects

# Demo: simulate objects moving across a frame
canvas = np.zeros((400,600,3),dtype=np.uint8)
tracker = CentroidTracker()
trails  = {}

# Simulate 3 moving objects
positions = [(50,100), (200,200), (400,300)]
colors    = [(0,255,0),(255,0,0),(0,0,255)]

for step in range(30):
    frame = canvas.copy()
    rects = []
    for i,(x,y) in enumerate(positions):
        nx = x + step*8 + np.random.randint(-3,3)
        ny = y + np.random.randint(-3,3)
        rects.append((nx-20,ny-20,40,40))
        cv2.rectangle(frame,(nx-20,ny-20),(nx+20,ny+20),colors[i],2)

    objects = tracker.update(rects)
    for oid, c in objects.items():
        if oid not in trails: trails[oid]=[]
        trails[oid].append(tuple(c))
        cv2.putText(frame,f'ID{oid}',tuple(c),cv2.FONT_HERSHEY_SIMPLEX,0.6,(255,255,0),2)
        if len(trails[oid])>1:
            for j in range(1,len(trails[oid])):
                cv2.line(frame,trails[oid][j-1],trails[oid][j],(255,255,0),1)

plt.imshow(cv2.cvtColor(frame,cv2.COLOR_BGR2RGB))
plt.title(f'Centroid tracker: {len(objects)} active objects with trails'); plt.axis('off'); plt.show()